# Get the unique skills list from job descriptions

In [3]:
# %%
import pandas as pd
# readthis file /Users/ryanlin/Downloads/job_des_full.csv
job_descriptions = pd.read_csv('/Users/ryanlin/Downloads/job_des_full.csv')
job_descriptions.head()
# for all the skills in the column 'skills_extracted' get the unique skills and make them a huge list
unique_skills = set()
for skills in job_descriptions['skills_extracted']:
    unique_skills.update(skills.split(','))
#show the unique skills
print(unique_skills)
print(len(unique_skills))

{" 'budget preparation'", " 'executive search'", " 'writing skills'", " 'heor'", " 'kpi development'", " 'constructive feedback'", " 'competitive analysis'", " 'social media'", " 'status reporting'", " 'client onboarding'", " 'data interfaces'", " 'scala'", " 'erp'", "['financial reporting'", " 'technical expertise'", "['interpersonal skills'", " 'supervisory experience'", " 'clinical trials'", " 'mef'", " 'technical analysis'", " 'network services'", " 'react'", "['api development'", " 'adf'", " 'configuration management'", " 'background checks'", " 'contract compliance'", " 'accounting standards'", " 'graphql'", " 'microsoft azure'", " 'hadoop'", "['healthcare technology'", " 'investigations'", " 'data collection'", "['financial transactions'", " 'networking technologies'", " 'journal entries'", "['process documentation'", "['administrative support'", " 'business management'", " 'coding standards'", " 'qlik'", " 'research'", "['profitability analysis'", " 'uml diagrams'", " 'performa

# build a resume skill extraction tool based on the unique skills extracted from job descriptions, using api of OpenAI

In [2]:
import pandas as pd

df = pd.read_csv("hf://datasets/opensporks/resumes/Resume/Resume.csv")
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [ ]:
from openai import OpenAI 

In [ ]:
# build a resume skill extraction tool based on the unique skills extracted from job descriptions, using api of OpenAI
from openai import OpenAI 

openai = OpenAI(api_key="")

# Extract ALL skills from the df dataframe for each row entry and turn it into a skills column
# Skills can be ANY technical or professional skills found in the resume
def extract_skills(resume_text):
    """
    Extract ALL skills from resume text using OpenAI API.
    Extracts any technical, professional, or soft skills mentioned.
    
    Args:
        resume_text: The resume text to extract skills from
    
    Returns:
        List of extracted skills (e.g., ['python', 'sql', 'excel', 'leadership'])
    """
    # Truncate resume if too long to avoid token limits
    max_resume_length = 4000
    resume_excerpt = resume_text[:max_resume_length] if len(resume_text) > max_resume_length else resume_text
    
    prompt = f"""Extract ALL skills from the following resume. Include technical skills, soft skills, tools, programming languages, frameworks, certifications, and professional competencies.

Resume Text:
{resume_excerpt}

IMPORTANT RULES:
1. Extract ALL relevant skills mentioned in the resume
2. Include technical skills (e.g., Python, SQL, Excel, AWS)
3. Include soft skills (e.g., leadership, communication, teamwork)
4. Include tools and technologies (e.g., SAP, Salesforce, Git)
5. Include certifications and methodologies (e.g., Agile, Scrum, PMP)
6. Return skills in lowercase format
7. Remove duplicates

Output Format: Return a Python list of all skills in lowercase, like this example:
['user training', 'leadership', 'management', 'sql', 'excel', 'python', 'communication']

If no skills found, return: []"""
    
    try:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",  # Using mini for cost-effectiveness
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert skill extraction assistant. Extract all technical and professional skills from resumes. Return a Python list format with lowercase skills."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,  # Deterministic output
            max_tokens=800  # Increased for more skills
        )
        
        # Parse the response to convert string representation to actual list
        result = response.choices[0].message.content.strip()
        
        # Convert string representation of list to actual Python list
        import ast
        try:
            skills_list = ast.literal_eval(result)
            # Remove duplicates while preserving order
            if isinstance(skills_list, list):
                seen = set()
                unique_skills = []
                for skill in skills_list:
                    skill_lower = skill.lower().strip()
                    if skill_lower not in seen:
                        seen.add(skill_lower)
                        unique_skills.append(skill_lower)
                return unique_skills
            return []
        except:
            # If parsing fails, try to extract skills manually
            if result and result != "[]":
                # Remove brackets and quotes, split by comma
                result = result.replace('[', '').replace(']', '').replace("'", "").replace('"', '')
                skills = [s.strip().lower() for s in result.split(',') if s.strip()]
                # Remove duplicates
                return list(dict.fromkeys(skills))
            return []
            
    except Exception as e:
        print(f"Error extracting skills: {e}")
        return []

# Test with first row to verify it works
print("Testing skill extraction with first resume...")
test_resume = df.iloc[0]['Resume_str']
test_result = extract_skills(test_resume)
print(f"Sample extraction result: {test_result}\n")
print(f"Number of skills extracted: {len(test_result)}\n")

Testing skill extraction with first resume...
Sample extraction result: ['```python\n\n    customer service', 'team management', 'marketing', 'conflict resolution', 'training and development', 'multi-tasking', 'client relations', 'microsoft office', 'customer loyalty', 'hospitality systems', 'hilton onq', 'micros', 'opera pms', 'fidelio', 'reservation system', 'holidex', 'sales strategies', 'inventory control', 'loss prevention', 'safety', 'time management', 'leadership', 'performance assessment', 'medical billing', 'icd-9', 'cpt', 'data analysis', 'billing standards', 'organizational skills', 'analytical skills', 'budgeting', 'financial management', 'accounting', 'human resources', 'payroll', 'purchasing', 'public relations', 'social media', 'website design', 'advertising', 'marketing collateral', 'employee benefits', 'employee relations', 'training', 'customer interaction\n\n```']

Number of skills extracted: 45



In [7]:
# Apply to all rows in the dataframe
# For cost efficiency, you might want to start with a smaller subset
print("Extracting skills from all resumes...")
print(f"Total resumes to process: {len(df)}")

# Process in batches with progress updates
from tqdm import tqdm
tqdm.pandas(desc="Extracting skills")

# Apply the function to create a new 'skills_extracted' column
df['skills_extracted'] = df['Resume_str'].progress_apply(lambda x: extract_skills(x))

# Show results
print(f"\nCompleted! Skills extracted for {len(df)} resumes.")
print("\nSample results:")
print(df[['ID', 'Category', 'skills_extracted']].head(10))

# Save results to avoid re-running
output_file = '/Users/ryanlin/Downloads/resumes_with_skills_2.csv'
df.to_csv(output_file, index=False)
print(f"\nResults saved to: {output_file}")



Extracting skills from all resumes...
Total resumes to process: 2484


Extracting skills: 100%|██████████| 2484/2484 [2:26:29<00:00,  3.54s/it]  



Completed! Skills extracted for 2484 resumes.

Sample results:
         ID Category                                   skills_extracted
0  16852973       HR  [```python\n\n    customer service, team manag...
1  22323967       HR  [```python\ncommunication, marketing, human re...
2  33176873       HR  [```python\nrecruiting, fmla, eeo, flsa, hris ...
3  27018550       HR  [```python\n10-key by touch, access, call moni...
4  17812897       HR  [```python\n\n    hr skills, hr department sta...
5  11592605       HR  [```python\nmicrosoft office, excel, attention...
6  25824789       HR  [```python\nadministrative duties, teambuildin...
7  15375009       HR  [```python\nadministrative, adp, backup, benef...
8  11847784       HR  [```python\nmanagement consultation, negotiati...
9  32896934       HR  [```python\napplicant tracking system, bookkee...

Results saved to: /Users/ryanlin/Downloads/resumes_with_skills_2.csv


In [ ]:
# some basic NER tasks
